# OmniVoice para AI Companions — Arranque automático

Ejecuta las celdas en orden. El notebook instala lo que falte, arranca el servidor compatible con AI Companions, inicia OpenPort y muestra la IPv4 y el puerto.

**Importante:** el código queda en GitHub, pero los pesos del modelo no. En una máquina Colab nueva puede ser necesario volver a descargar el modelo. La sesión de Colab debe permanecer activa mientras juegas.

In [ ]:
import importlib.util, subprocess, sys

required = ['omnivoice', 'fastapi', 'uvicorn', 'soundfile', 'requests']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print('Instalando:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'omnivoice', 'fastapi', 'uvicorn', 'soundfile', 'requests'])
else:
    print('Dependencias ya instaladas.')


In [ ]:
from pathlib import Path
server_code = 'import io\nimport re\nimport time\nimport uuid\nimport hashlib\nimport threading\nfrom collections import OrderedDict\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom fastapi import FastAPI, Request, HTTPException\nfrom fastapi.responses import Response, JSONResponse\nfrom omnivoice import OmniVoice\nfrom omnivoice.models.omnivoice import OmniVoiceGenerationConfig\nimport uvicorn\n\nMODEL_NAME = "k2-fsa/OmniVoice"\nHOST = "0.0.0.0"\nPORT = 8000\nMAX_PROFILES = 8\nMAX_CLIP_BYTES = 30 * 1024 * 1024\nSERVICE = "ai-npc-omnivoice-server"\nPROTOCOL = 1\nINSTANCE = str(uuid.uuid4())\nVERSION = "1.0.0"\n\napp = FastAPI()\nmodel = None\nmodel_ready = False\nmodel_error = None\nclips = {}\nvoice_table = {}\nprompts = OrderedDict()\nmodel_lock = threading.Lock()\n\ndef load_model():\n    global model, model_ready, model_error\n    try:\n        print("Cargando OmniVoice...", flush=True)\n        model = OmniVoice.from_pretrained(\n            MODEL_NAME,\n            device_map="cuda:0" if torch.cuda.is_available() else "cpu",\n            dtype=torch.float16 if torch.cuda.is_available() else torch.float32\n        )\n        model_ready = True\n        print("OmniVoice listo.", flush=True)\n    except Exception as e:\n        model_error = str(e)\n        print("ERROR cargando OmniVoice:", repr(e), flush=True)\n\ndef valid_key(key):\n    return isinstance(key, str) and bool(re.fullmatch(r"[a-z0-9][a-z0-9-]{0,79}", key))\n\ndef get_descriptor(profile):\n    descriptor = voice_table.get(profile)\n    if not isinstance(descriptor, dict):\n        raise HTTPException(404, detail={"code": "unknown-profile", "message": "Perfil desconocido"})\n    if "reference" not in descriptor:\n        raise HTTPException(404, detail={"code": "unknown-profile", "message": "Perfil sin referencia"})\n    return descriptor\n\ndef get_clip(profile):\n    descriptor = get_descriptor(profile)\n    clip = clips.get(profile)\n    if not clip or clip["sha256"] != descriptor.get("sha256"):\n        raise HTTPException(404, detail={"code": "unknown-profile", "message": "Falta el audio de referencia"})\n    return clip["bytes"], descriptor\n\ndef build_prompt(profile):\n    audio_bytes, descriptor = get_clip(profile)\n    digest = descriptor["sha256"]\n    cached = prompts.get(profile)\n    if cached and cached["sha256"] == digest:\n        prompts.move_to_end(profile)\n        return cached["prompt"]\n    try:\n        audio, sample_rate = sf.read(io.BytesIO(audio_bytes), dtype="float32", always_2d=False)\n        if audio.ndim == 2:\n            audio = audio.mean(axis=1)\n        waveform = torch.from_numpy(np.asarray(audio, dtype=np.float32).copy())\n        prompt = model.create_voice_clone_prompt(\n            ref_audio=(waveform, sample_rate),\n            ref_text=descriptor.get("transcript", ""),\n            preprocess_prompt=True\n        )\n        prompts[profile] = {"sha256": digest, "prompt": prompt}\n        prompts.move_to_end(profile)\n        while len(prompts) > MAX_PROFILES:\n            prompts.popitem(last=False)\n        return prompt\n    except HTTPException:\n        raise\n    except Exception as e:\n        print("Error preparando perfil:", profile, repr(e), flush=True)\n        raise HTTPException(500, detail={"code": "voice-prepare-failed", "message": str(e)})\n\ndef error_response(code, message, status):\n    return JSONResponse(status_code=status, content={"error": {"code": code, "message": message}})\n\n@app.get("/v1/health")\ndef health():\n    if model_error:\n        return {"service": SERVICE, "protocol": PROTOCOL, "state": "error",\n                "detail": model_error[:120], "instance": INSTANCE, "version": VERSION}\n    return {"service": SERVICE, "protocol": PROTOCOL,\n            "state": "ready" if model_ready else "loading",\n            "instance": INSTANCE,\n            "device": "cuda" if torch.cuda.is_available() else "cpu",\n            "version": VERSION}\n\n@app.post("/v1/voices")\nasync def receive_voices(request: Request):\n    global voice_table\n    data = await request.json()\n    incoming = data.get("voices", {})\n    if not isinstance(incoming, dict):\n        return error_response("invalid-voices", "Tabla de voces no válida", 400)\n    missing = []\n    new_table = {}\n    for key, descriptor in incoming.items():\n        if not valid_key(key) or not isinstance(descriptor, dict):\n            continue\n        reference = descriptor.get("reference", "")\n        sha = descriptor.get("sha256", "")\n        transcript = descriptor.get("transcript", "")\n        if not isinstance(reference, str) or not reference.startswith(\n            ("runtime/voice-references/", "runtime/player-voices/")\n        ):\n            continue\n        if not re.fullmatch(r"[a-f0-9]{64}", sha):\n            continue\n        if not isinstance(transcript, str) or not transcript.strip() or len(transcript) > 12000:\n            continue\n        new_table[key] = {"reference": reference, "sha256": sha, "transcript": transcript}\n        clip = clips.get(key)\n        if not clip or clip["sha256"] != sha:\n            missing.append({"key": key})\n    for key in list(voice_table):\n        if key not in new_table:\n            prompts.pop(key, None)\n            clips.pop(key, None)\n    voice_table = new_table\n    return {"missing": missing, "prepared": list(prompts.keys())}\n\n@app.put("/v1/clips/{key}")\nasync def upload_clip(key: str, request: Request):\n    if not valid_key(key):\n        raise HTTPException(400, "Clave de perfil inválida")\n    if key not in voice_table:\n        raise HTTPException(404, "Perfil desconocido")\n    expected = request.query_params.get("sha256", "")\n    if expected != voice_table[key]["sha256"]:\n        raise HTTPException(400, "SHA256 no coincide")\n    audio_bytes = await request.body()\n    if not audio_bytes or len(audio_bytes) > MAX_CLIP_BYTES:\n        raise HTTPException(413, "Audio vacío o demasiado grande")\n    actual = hashlib.sha256(audio_bytes).hexdigest()\n    if actual != expected:\n        raise HTTPException(400, "SHA256 incorrecto")\n    try:\n        sf.info(io.BytesIO(audio_bytes))\n    except Exception:\n        raise HTTPException(400, "WAV no válido")\n    clips[key] = {"sha256": actual, "bytes": audio_bytes}\n    prompts.pop(key, None)\n    return {"ok": True, "key": key}\n\n@app.post("/v1/prepare")\nasync def prepare_voice(request: Request):\n    if not model_ready:\n        return error_response("server-loading", "OmniVoice todavía está cargando", 503)\n    data = await request.json()\n    profile = data.get("profile")\n    steps = data.get("steps", 32)\n    if not valid_key(profile):\n        return error_response("invalid-profile", "Perfil inválido", 400)\n    if steps not in (16, 32):\n        return error_response("invalid-steps", "Calidad inválida", 400)\n    started = time.perf_counter()\n    try:\n        with model_lock:\n            build_prompt(profile)\n        return {"prepared": profile, "ms": round((time.perf_counter() - started) * 1000, 2)}\n    except HTTPException as e:\n        return JSONResponse(status_code=e.status_code, content=e.detail)\n\n@app.post("/v1/synthesize")\nasync def synthesize(request: Request):\n    if not model_ready:\n        return error_response("server-loading", "OmniVoice todavía está cargando", 503)\n    data = await request.json()\n    profile = data.get("profile")\n    text = data.get("text", "")\n    steps = data.get("steps", 32)\n    language = data.get("language")\n    if not valid_key(profile):\n        return error_response("invalid-profile", "Perfil inválido", 400)\n    if not isinstance(text, str) or not text.strip() or len(text) > 12000:\n        return error_response("invalid-text", "Texto inválido", 400)\n    if steps not in (16, 32):\n        return error_response("invalid-steps", "Calidad inválida", 400)\n    if language is not None and (\n        not isinstance(language, str) or not re.fullmatch(r"[a-z]{2,3}", language)\n    ):\n        return error_response("invalid-language", "Idioma inválido", 400)\n    started = time.perf_counter()\n    try:\n        with model_lock:\n            prompt = build_prompt(profile)\n            generation_started = time.perf_counter()\n            kwargs = {\n                "text": text,\n                "voice_clone_prompt": prompt,\n                "generation_config": OmniVoiceGenerationConfig(num_step=steps)\n            }\n            if language:\n                kwargs["language"] = language\n            result = model.generate(**kwargs)\n            generation_ms = (time.perf_counter() - generation_started) * 1000\n            if not result:\n                raise RuntimeError("OmniVoice no generó audio")\n            audio = np.asarray(result[0], dtype=np.float32)\n            if audio.ndim > 1:\n                audio = np.squeeze(audio)\n            wav_buffer = io.BytesIO()\n            sf.write(wav_buffer, audio, model.sampling_rate, format="WAV", subtype="PCM_16")\n            wav_bytes = wav_buffer.getvalue()\n        timing = {\n            "prepared": profile,\n            "voiceGenerationMs": round(generation_ms, 2),\n            "voiceRuntime": "omnivoice",\n            "voiceAcceleration": "eager",\n            "voiceDtype": "float16" if torch.cuda.is_available() else "float32"\n        }\n        return Response(\n            content=wav_bytes,\n            media_type="audio/wav",\n            headers={"X-OmniVoice-Timing": __import__("json").dumps(timing)}\n        )\n    except HTTPException as e:\n        return JSONResponse(status_code=e.status_code, content=e.detail)\n    except Exception as e:\n        print("ERROR SYNTHESIS:", repr(e), flush=True)\n        return error_response("synthesis-failed", str(e)[:500], 500)\n\nif __name__ == "__main__":\n    load_model()\n    uvicorn.run(app, host=HOST, port=PORT, log_level="info")\n'
server_path = Path('/content/servidor_omnivoice.py')
server_path.write_text(server_code, encoding='utf-8')
print('Servidor preparado:', server_path)


In [ ]:
import os, re, socket, subprocess, sys, time, requests
from pathlib import Path

BASE = 'http://127.0.0.1:8000'
server_log = open('/content/omnivoice-server.log', 'a', buffering=1)

# 1) Reutilizar el servidor si ya está activo; si no, arrancarlo.
try:
    current = requests.get(BASE + '/v1/health', timeout=3).json()
except Exception:
    current = None

if current is None:
    server_proc = subprocess.Popen(
        [sys.executable, '-u', '/content/servidor_omnivoice.py'],
        stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True
    )
    print('Servidor iniciado. PID:', server_proc.pid)
else:
    print('Servidor ya estaba activo:', current)

# 2) Esperar a que OmniVoice termine de cargar.
print('Esperando a OmniVoice (puede tardar si debe descargar los pesos)...')
health = None
for _ in range(900):
    try:
        health = requests.get(BASE + '/v1/health', timeout=5).json()
        if health.get('state') == 'ready':
            break
        if health.get('state') == 'error':
            raise RuntimeError('Error cargando OmniVoice: ' + str(health.get('detail')))
    except requests.RequestException:
        pass
    time.sleep(2)
if not health or health.get('state') != 'ready':
    print(Path('/content/omnivoice-server.log').read_text(errors='ignore')[-5000:])
    raise RuntimeError('OmniVoice no llegó a estado ready.')
print('OmniVoice LISTO:', health)

# 3) Instalar OpenPort solo si no existe.
openport_bin = '/root/.openport/bin/openport'
if not os.path.isfile(openport_bin):
    print('Instalando OpenPort...')
    subprocess.run('curl -sSf https://openport.pw/install.sh | sh', shell=True, check=True)

# 4) Reutilizar túnel vivo si existe; de lo contrario, crear uno.
tunnel_log_path = '/content/openport-server.log'
tunnel_log = open(tunnel_log_path, 'a', buffering=1)
try:
    tunnel_check = subprocess.run(['pgrep', '-af', '/root/.openport/bin/openport'], capture_output=True, text=True)
    tunnel_alive = bool(tunnel_check.stdout.strip())
except Exception:
    tunnel_alive = False
if not tunnel_alive:
    tunnel_proc = subprocess.Popen(
        [openport_bin, '8000', '--tcp'],
        stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True
    )
    print('OpenPort iniciado. PID:', tunnel_proc.pid)
else:
    print('Ya hay un proceso OpenPort activo.')

# 5) Leer la asignación de puerto.
forward = None
for _ in range(30):
    try:
        log_text = Path(tunnel_log_path).read_text(errors='ignore')
    except Exception:
        log_text = ''
    matches = re.findall(r'Forwarding\s+openport\.pw:(\d+)\s+->\s+127\.0\.0\.1:8000', log_text)
    if matches:
        forward = matches[-1]
        break
    time.sleep(1)
if not forward:
    print(Path(tunnel_log_path).read_text(errors='ignore')[-3000:])
    raise RuntimeError('No se encontró la asignación del túnel OpenPort.')

# 6) Resolver IPv4 y comprobar el acceso por el túnel.
ip = socket.gethostbyname('openport.pw')
port = int(forward)
external = f'http://{ip}:{port}/v1/health'
response = requests.get(external, timeout=20)
remote_health = response.json()
if response.status_code != 200 or remote_health.get('service') != 'ai-npc-omnivoice-server' or remote_health.get('state') != 'ready':
    raise RuntimeError(f'El túnel no superó la comprobación: HTTP {response.status_code} {remote_health}')

print('\n' + '='*54)
print('       OMNIVOICE LISTO PARA AI COMPANIONS')
print('='*54)
print('IPv4 :', ip)
print('Puerto:', port)
print('En el mod introduce solo esos dos valores.')
print('No cierres la sesión de Colab mientras juegas.')
print('='*54)


## Si algo falla

Consulta los registros con:

```python
print(open('/content/omnivoice-server.log').read()[-8000:])
print(open('/content/openport-server.log').read()[-3000:])
```

Si Colab ha terminado completamente la máquina virtual, el modelo puede tener que descargarse otra vez. GitHub conserva este notebook y el código del servidor, no los procesos ni la memoria GPU.